In [1]:
import duckdb
import pandas as pd
import json

print("Initializing DuckDB and generating complete TalkTalk sample extracts...")

# Connect to DuckDB
con = duckdb.connect(database=':memory:')

# ==========================================
# 1. ESTABLISH THE ANCHOR SAMPLE
# ==========================================
# Extract 50 random customer IDs to act as our universal filter
con.execute("""
    CREATE TEMP TABLE sample_ids AS
    SELECT unique_customer_identifier 
    FROM read_parquet('customer_info.parquet') 
    USING SAMPLE 50 ROWS;
""")

# ==========================================
# 2. EXTRACT DATA MAINTAINING INTEGRITY
# ==========================================
print("Extracting customer_info sample...")
customer_sample_df = con.execute("""
    SELECT * FROM read_parquet('customer_info.parquet')
    WHERE unique_customer_identifier IN (SELECT unique_customer_identifier FROM sample_ids)
""").df()
customer_sample_df.to_csv('lovable_sample_customer_info.csv', index=False)

print("Extracting usage sample...")
usage_sample_df = con.execute("""
    SELECT * FROM read_parquet('usage.parquet')
    WHERE unique_customer_identifier IN (SELECT unique_customer_identifier FROM sample_ids)
""").df()
usage_sample_df.to_csv('lovable_sample_usage.csv', index=False)

print("Extracting calls sample...")
# Note: Using sample_size=-1 to bypass CSV type-guessing errors
calls_sample_df = con.execute("""
    SELECT * FROM read_csv_auto('calls.csv', sample_size=-1, ignore_errors=true)
    WHERE unique_customer_identifier IN (SELECT unique_customer_identifier FROM sample_ids)
""").df()
calls_sample_df.to_csv('lovable_sample_calls.csv', index=False)

print("Extracting cease sample...")
cease_sample_df = con.execute("""
    SELECT * FROM read_csv_auto('cease.csv', sample_size=-1, ignore_errors=true)
    WHERE unique_customer_identifier IN (SELECT unique_customer_identifier FROM sample_ids)
""").df()
cease_sample_df.to_csv('lovable_sample_cease.csv', index=False)

# ==========================================
# 3. EXTRACT FULL SCHEMAS FOR LOVABLE
# ==========================================
print("Extracting schema definitions for all tables...")
schemas = {
    "customer_info": "DESCRIBE SELECT * FROM read_parquet('customer_info.parquet')",
    "usage": "DESCRIBE SELECT * FROM read_parquet('usage.parquet')",
    "calls": "DESCRIBE SELECT * FROM read_csv_auto('calls.csv', sample_size=-1, ignore_errors=true)",
    "cease": "DESCRIBE SELECT * FROM read_csv_auto('cease.csv', sample_size=-1, ignore_errors=true)"
}

schema_definition = {}
for table_name, query in schemas.items():
    schema_df = con.execute(query).df()
    schema_definition[f"Table: {table_name}"] = {
        row['column_name']: row['column_type'] 
        for _, row in schema_df.iterrows()
    }

# Save the schema to a text file for your master prompt
with open('lovable_full_schema_types.txt', 'w') as f:
    f.write("TALKTALK COMPLETE DATA MODEL SCHEMA:\n\n")
    f.write(json.dumps(schema_definition, indent=4))

print("✅ Complete! Check your folder for the four CSVs and the Full Schema text file.")

Initializing DuckDB and generating complete TalkTalk sample extracts...
Extracting customer_info sample...
Extracting usage sample...
Extracting calls sample...
Extracting cease sample...
Extracting schema definitions for all tables...
✅ Complete! Check your folder for the four CSVs and the Full Schema text file.
